# VSAD 0.0.3 — Local Runtime Manual Test

Notebook chỉ dùng runtime local trong project. Không phụ thuộc workspace huấn luyện `C:\Users\ASUS\Transformer`.

In [1]:
from __future__ import annotations
import csv, json, sys
from datetime import datetime
from pathlib import Path
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "Runtime").is_dir():
    raise FileNotFoundError("Hãy chạy notebook từ thư mục gốc AL_voice_local")
sys.path.insert(0, str(ROOT))

from Runtime.model import VSADModel

RELEASE = ROOT / "Runtime/Model/VSAD/0.0.3"
CSV_PATH = ROOT / "Artifacts/vsad_0.0.3_manual_tests.csv"
vsad = VSADModel(RELEASE)
print({"release": "VSAD 0.0.3", "device": str(vsad.device), "model_dir": str(RELEASE), "csv": str(CSV_PATH)})


{'release': 'VSAD 0.0.3', 'device': 'cuda', 'model_dir': 'C:\\Users\\ASUS\\AL_voice_local\\Runtime\\Model\\VSAD\\0.0.3', 'csv': 'C:\\Users\\ASUS\\AL_voice_local\\Artifacts\\vsad_0.0.3_manual_tests.csv'}


In [2]:
def _infer(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text phải là chuỗi không rỗng")
    result = vsad.infer(text.strip())
    return {
        "response": result["response"],
        "parameters": result["parameters"],
        "act": result["act"],
        "goal": result["goal"],
    }

def _append_csv(path: Path, text: str, result: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists() or path.stat().st_size == 0
    with path.open("a", encoding="utf-8-sig", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["timestamp", "input", "response", "parameters", "act", "goal"])
        if write_header:
            writer.writeheader()
        writer.writerow({
            "timestamp": datetime.now().astimezone().isoformat(timespec="seconds"),
            "input": text,
            "response": result["response"],
            "parameters": json.dumps(result["parameters"], ensure_ascii=False, separators=(",", ":")),
            "act": result["act"],
            "goal": result["goal"],
        })

def test(text: str, csv_path: Path = CSV_PATH) -> dict:
    result = _infer(text)
    _append_csv(csv_path, text, result)
    display(result)
    return result


In [3]:
# Mỗi lần gọi sẽ hiển thị kết quả và append một dòng vào CSV_PATH.
result = test("Khởi động lại máy giúp tôi")


{'response': 'Bạn có chắc khởi động lại hệ thống không?',
 'parameters': {'command_id': 'RESTART_SYSTEM'},
 'act': 'ASK_CLARIFICATION',
 'goal': 'RUN_COMMAND'}